## Settings & imports

In [1]:
import numpy as np
import pandas as pd
import os
import torch
import matplotlib.pyplot as plt
import string
from torch import nn
from torch.utils.data import Dataset, DataLoader
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize
import csv

In [2]:
object_name = 'Konstytucja'

In [3]:
keep_sample_together = False #Should all 15 slices from the same sample go together to the same set (train/test)?
if keep_sample_together:
    sample_together = 'sample_together'
else:
    sample_together = 'sample_split'
    
preprocessing_method = 'logarithm' # none / normalization / logarithm

remove_outer = True
groups_to_remove = [0, 14]

columns_to_keep_inds = { 
                        'Konstytucja' : [
                                                'Al_inds',
                                                'S_inds',
                                                'Cr_inds',
                                                'Mn_inds',
                                                'Co_inds',
                                                'Cu_inds',
                                                'Zn_inds',
                                                'Pb_inds'
                        ]
                        }

input_size = len(columns_to_keep_inds['Konstytucja'])

In [4]:
data_path = {'Konstytucja' : '../data/Listopad_2024.xlsx'}

results_path = {'Konstytucja' : '../results/Konstytucja/'}

figures_path = {'Konstytucja': '../results/visualisations/Konstytucja/'}

models_path = {'Konstytucja' : '../models/Documents_XVIII_century/model_regression_Documents_XVIII_century_sample_split_logarithm_outer_removed_Al_S_Cr_Mn_Co_Cu_Zn_Pb_2024_12_07_23_35_27'}

## Loading the data

In [5]:
df = pd.ExcelFile(data_path[object_name])
    
if object_name == 'Konstytucja':
    inds_df = df.parse('Arkusz1', header=0, usecols=range(32,59), skiprows=range(1, 3825))
    inds_df.columns = [x.split('.')[0]+'_inds' for x in list(inds_df.columns)]
    inds_df.drop(index=[0], inplace=True)

## Preprocessing

In [6]:
inds_df = inds_df.reset_index(drop=True)

### Removing some data

1. Let's remove 'outer' samples.

In [7]:
if remove_outer:
    for group_nr in groups_to_remove:
        inds_df = inds_df[inds_df.index%15 != group_nr]

2. Let's keep only columns that we need.

To reduce the set of used elements run cell below. Then, instead of predicting 29 numbers, we will predict only 8. We will also use only 8 numbers as input.

In [8]:
inds_df = inds_df[columns_to_keep_inds[object_name]]

2. Let's remove rows with missing values.

In [9]:
(inds_df.shape[0] - inds_df.dropna().shape[0])/inds_df.shape[0]

0.0

In [10]:
inds_df.dropna(inplace=True)

In [11]:
X = np.array(inds_df.values)

In [12]:
######################################################

### Normalizing / taking logarithm

In [13]:
def adjusted_log_transform(nonnegative_array):
    res = np.where(nonnegative_array>0, np.log(nonnegative_array), 0.)
    res = np.where(res != 0, res, 2*res.min(axis=0))
    return res

In [14]:
if preprocessing_method == 'normalization':

    X = (X - np.min(X, axis=0))/np.std(X, axis=0)
    
elif preprocessing_method == 'logarithm':
    
    X = adjusted_log_transform(X)
    
elif preprocessing_method == 'none':
    
    pass

In [15]:
# coor = 0
# plt.hist(X[:,coor], bins=100)

### Converting to tensors

In [16]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

Using cuda device


In [17]:
X = torch.Tensor(X).to(device)

## Loading the trained model

### Model class

In [19]:
dropout_prob = 0.02

class InksNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.seq = nn.Sequential(
        nn.Linear(input_size, 32),
        nn.Dropout(dropout_prob),
        nn.ReLU(),
        nn.Linear(32, 64),
        nn.Dropout(dropout_prob),
        nn.ReLU(),
        nn.Linear(64, 128),
        nn.Dropout(dropout_prob),
        nn.ReLU(),
        nn.Linear(128, 256),
        nn.Dropout(dropout_prob),
        nn.ReLU(),
        nn.Linear(256, 128),
        nn.Dropout(dropout_prob),
        nn.ReLU(),
        nn.Linear(128, 64),
        nn.Dropout(dropout_prob),
        nn.ReLU(),
        nn.Linear(64, 32),
        nn.Dropout(dropout_prob),
        nn.ReLU(),
        nn.Linear(32, input_size))
    def forward(self, x):
        return self.seq(x)

### Loading

In [20]:
model = InksNet().to(device)
model.load_state_dict(torch.load(models_path[object_name]))

/tmp/ipykernel_10609/30850976.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(models_path[object_name]))


<All keys matched successfully>

## Prediction

In [21]:
model.eval()
outputs = model(X)

## Saving result to a file

In [22]:
model_name = models_path[object_name].split('/')[-1]

In [23]:
outputs_to_save = outputs.cpu().detach().numpy()

In [24]:
np.savetxt(results_path[object_name] + 'prediction_from_' + model_name, outputs_to_save, delimiter=',')